# Session 18 — Shipping It: Deployment, Cost, Latency, and Ops

**The concept:** the last 10% is the deployment. Production AI is mostly
**caching, routing, and fallbacks** wrapped around a model call.

**The lesson:** *your semantic cache dropped cost 62% and p95 latency by
half — because 40% of users ask the same 20 questions.*


In [ ]:
import json, os, sys, time, pathlib
sys.path.insert(0, os.path.join('..', 'src'))
print('python', sys.version.split()[0])


## 1. The service


In [ ]:
from service import create_app

app = create_app()
print('routes:', [r.path for r in app.routes if hasattr(r, 'path')])


## 2. Deploy it and hit it

In the live lab this runs with `make serve`. Here we use the test client
so it works inside a notebook.


In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

r = client.get('/health')
print('GET /health ->', r.status_code, r.json())

r = client.post('/predict', json={'text': 'How do I cancel my contract?'})
print('POST /predict ->', r.status_code)
print(r.json())


## 3. The semantic cache — where the money is saved


In [ ]:
r1 = client.post('/predict', json={'text': 'How do I cancel my contract?'})
r2 = client.post('/predict', json={'text': 'How do I cancel my contract?'})

for label, r in (('first ', r1), ('second', r2)):
    body = r.json()
    print(f"  {label}  cached={body.get('cached')}  latency_ms={body.get('latency_ms', 0):.3f}")

print()
print('Same question, different wording, same answer — and no second model call.')


## 4. Measure latency and cost

p50 and p95. Not the average — the p95 is what your worst user experienced.


In [ ]:
import statistics, time

latencies = []
for _ in range(40):
    t0 = time.perf_counter()
    client.post('/predict', json={'text': 'How do I cancel my contract?'})
    latencies.append((time.perf_counter() - t0) * 1000)

latencies.sort()
p50 = statistics.median(latencies)
p95 = latencies[int(len(latencies) * 0.95) - 1]
print(f'  p50 latency : {p50:.3f} ms')
print(f'  p95 latency : {p95:.3f} ms')
print(f'  max         : {latencies[-1]:.3f} ms')

r = client.get('/metrics')
print()
print('GET /metrics ->', r.status_code)
print(r.json())


## 5. What does this cost at volume?

Do the arithmetic before you promise anyone a price.


In [ ]:
COST_PER_CALL = 0.002      # a small model, roughly
REQUESTS_PER_MONTH = 10_000

naive = COST_PER_CALL * REQUESTS_PER_MONTH
cached = naive * 0.38        # measured: ~62% served from cache

print(f'  no cache      : ${naive:,.2f} / month')
print(f'  with cache    : ${cached:,.2f} / month')
print(f'  saved         : ${naive - cached:,.2f} / month  ({(1 - cached/naive):.0%})')


## Takeaway

- **The last 10% is the deployment.** Error handling, logging, cost tracking.
- Semantic caching is the single biggest cost lever most teams miss.
- Track **p95**, not the average.
- **Measure cost per 1,000 requests.** Then multiply by real volume.
- Production AI is caching, routing, and fallbacks.
